# DX 704 Week 4 Project

This week's project will test the learning speed of linear contextual bandits compared to unoptimized approaches.
You will start with building a preference data set for evaluation, and then implement different variations of LinUCB and visualize how fast they learn the preferences.


The full project description, a template notebook and supporting code are available on GitHub: [Project 4 Materials](https://github.com/bu-cds-dx704/dx704-project-04).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Collect Rating Data

The file "recipes.tsv" in this repository has information about 100 recipes.
Make a new file "ratings.tsv" with two columns, recipe_slug (from recipes.tsv) and rating.
Populate the rating column with values between 0 and 1 where 0 is the worst and 1 is the best.
You can assign these ratings however you want within that range, but try to make it reflect a consistent set of preferences.
These could be your preferences, or a persona of your choosing (e.g. chocolate lover, bacon-obsessed, or sweet tooth).
Make sure that there are at least 10 ratings of zero and at least 10 ratings of one.


Hint: You may find it more convenient to assign raw ratings from 1 to 5 and then remap them as follows.

`ratings["rating"] = (ratings["rating_raw"] - 1) * 0.25`

In [8]:
import pandas as pd
import numpy as np

recipes = pd.read_csv(r'..\dx704-project-04\recipes.tsv', sep='\t')
recipes

,recipe_slug,recipe_title,recipe_introduction
0,falafel,Falafel,Falafel is a popular Middle Eastern dish made ...
1,spamburger,Spamburger,Spamburger is a type of hamburger that is made...
2,bacon-fried-rice,Bacon Fried Rice,Bacon fried rice is a savory and satisfying di...
3,chicken-fingers,Chicken Fingers,Chicken fingers are a popular dish made from c...
4,apple-crisp,Apple Crisp,Apple crisp is a classic dessert made with bak...
...,...,...,...
95,bacon-mac-and-cheese,Bacon Mac And Cheese,Bacon mac and cheese is a delicious and comfor...
96,chicken-alfredo-lasagna,Chicken Alfredo Lasagna,Chicken alfredo lasagna is a delicious twist o...
97,classic-beef-lasagna,Classic Beef Lasagna,Classic beef lasagna is a hearty and comfortin...
98,vegetarian-mushroom-lasagna,Vegetarian Mushroom Lasagna,Vegetarian mushroom lasagna is a delicious and...


In [16]:
food_rating = {
    'falafel': 0.8,
    'spamburger': 0.0,
    'bacon-fried-rice': 0.6,
    'chicken-fingers': 0.35,
    'apple-crisp': 0.85,
    'cranberry-apple-crisp': 0.8,
    'bacon-chocolate-chip-cookies': 0.1,
    'sujebi': 0.5,
    'pasta-primavera': 0.55,
    'ramen': 0.75,
    'brownies': 0.7,
    'bacon-wrapped-scallops': 0.55,
    'bacon-egg-muffins': 0.6,
    'breakfast-burritos': 0.2,
    'bolognese-sauce': 0.65,
    'croissants': 0.75,
    'bacon-souffle': 0.55,
    'souffle': 0.5,
    'chocolate-souffle': 0.5, 
    'butter-croissants': 0.75,
    'pickled-green-beans': 0.25,
    'fried-oysters': 0.755,
    'peanut-butter-brownies': 0.8,
    'almond-chip-cookies': 0.77,
    'guacamole': 0.8,
    'brioche-bread': 0.65,
    'brioche-bread-with-chocolate': 0.7,
    'asparagus-quiche': 0.2,
    'asparagus-burger': 0.05,
    'ma-la-chicken': 0.5, 
    'cold-noodles-with-tomatoes': 0.2,
    'tamales': 0.9,
    'enchiladas': 0.915,
    'tacos': 0.95,
    'burritos': 0.925,
    'peanut-butter-cupcakes': 0.7,
    'pickled-asparagus': 0.15,
    'chocolate-peanut-butter-cake': 0.6,
    'udon': 0.78,
    'vegetable-lasagna': 0.68,
    'maple-bacon-donuts': 0.625,
    'maple-bacon-pancakes': 0.645,
    'spinach-and-ricotta-stuffed-shells': 0.78,
    'quesadillas': 0.58,
    'pico-de-gallo': 0.6,
    'nachos': 0.72,
    'cold-soba-noodles-with-tomato-and-cucumber': 0.5,
    'fajitas': 0.82,
    'loaded-nachos': 0.75,
    'lasagna': 0.76,
    'chicken-nuggets': 0.5,
    'pain-au-chocolat': 0.83,
    'nacho-dip': 0.7,
    'nacho-salad': 0.7,
    'nacho-casserole': 0.67,
    'nacho-fries': 0.72,
    'tempura-udon': 0.78,
    'peach-crisp': 0.82,
    'apple-pie': 0.85,
    'blueberry-crisp': 0.8,
    'berry-crumble': 0.75,
    'strawberry-rhubarb-crisp': 0.9,
    'peach-cobbler': 0.83,
    'cranberry-sauce': 0.33, 
    'cherry-cobbler': 0.79,
    'apple-crumble': 0.825,
    'chocolate-croissants': 0.83,
    'french-toast': 0.45,
    'chocolate-babka': 0.65,
    'cinnamon-babka': 0.67, 
    'blueberry-crumble': 0.74,
    'spam-musubi': 0.5,
    'taco-salad': 0.52,
    'chiles-rellenos': 0.79,
    'almond-croissants': 0.65,
    'cranberry-relish': 0.05,
    'ma-la-beef': 0.45,
    'chocolate-cake': 0.57,
    'spinach-quiche': 0.565,
    'mushroom-quiche': 0.43,
    'quiche-lorraine': 0.5,
    'croissant-aux-amandes': 0.71,
    'danishes': 0.35,
    'spinach-and-feta-quiche': 0.68,
    'cold-sesame-noodles': 0.705,
    'soba-noodle-salad-with-peanut-dressing': 0.725,
    'dan-dan-noodles': 0.86,
    'vanilla-souffle': 0.6,
    'bacon-wrapped-dates': 0.25,
    'cherry-pie': 0.75,
    'berry-crisp': 0.715,
    'bacon-and-egg-breakfast-sandwich': 0.685,
    'bacon-wrapped-shrimp-skewers': 0.625,
    'bacon-wrapped-chicken': 0.635,
    'bacon-wrapped-asparagus': 0.615,
    'bacon-mac-and-cheese': 0.785,
    'chicken-alfredo-lasagna': 0.655,
    'classic-beef-lasagna': 0.685,
    'vegetarian-mushroom-lasagna': 0.455,
    'spinach-and-ricotta-lasagna': 0.755
}

ratings = recipes[['recipe_slug']].copy()

ratings['rating'] = recipes['recipe_slug'].map(food_rating)
ratings

,recipe_slug,rating
0,falafel,0.800
1,spamburger,0.000
2,bacon-fried-rice,0.600
3,chicken-fingers,0.350
4,apple-crisp,0.850
...,...,...
95,bacon-mac-and-cheese,0.785
96,chicken-alfredo-lasagna,0.655
97,classic-beef-lasagna,0.685
98,vegetarian-mushroom-lasagna,0.455


In [18]:
ratings.to_csv('ratings.tsv', sep='\t', index=False)

Submit "ratings.tsv" in Gradescope.

## Part 2: Construct Model Input

Use your file "ratings.tsv" combined with "recipe-tags.tsv" to create a new file "features.tsv" with a column recipe_slug, a column bias which is hard-coded to one, and a column for each tag that appears in "recipe-tags.tsv".
The tag column in this file should be a 0-1 encoding of the recipe tags for each recipe.
[Pandas reshaping function methods](https://pandas.pydata.org/docs/user_guide/reshaping.html) may be helpful.

The bias column will make later LinUCB calculations easier since it will just be another dimension.

Hint: For later modeling steps, it will be important to have the feature data (inputs) and the rating data (target outputs) in the same order.
It is highly recommended to make sure that "features.tsv" and "ratings.tsv" have the recipe slugs in the same order.

In [19]:
recipe_tags = pd.read_csv('recipe-tags.tsv', sep='\t')
recipe_tags.head()

,recipe_slug,recipe_tag
0,spam-musubi,hawaiian
1,spam-musubi,nori
2,spam-musubi,onthego
3,spam-musubi,rice
4,spam-musubi,snack


In [ ]:
tags = (
    pd.crosstab(recipe_tags['recipe_slug'], recipe_tags['recipe_tag']).clip(upper=1).reset_index()
)

tags

recipe_tag,recipe_slug,alfredo,almond,american,appetizer,appetizers,apple,asiancuisine,asparagus,avocado,...,udonnoodles,vanilla,vanillaicecream,vegan,vegetables,vegetarian,warm,whippedcream,winter,yeastdough
0,almond-chip-cookies,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,almond-croissants,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,apple-crisp,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,1,0
3,apple-crumble,0,0,0,0,0,1,0,0,0,...,0,0,1,0,0,0,0,1,0,0
4,apple-pie,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,tempura-udon,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
96,udon,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,1,0,0,0,0
97,vanilla-souffle,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
98,vegetable-lasagna,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0


In [21]:
features = pd.merge(ratings, tags, on='recipe_slug', how='left')
features

,recipe_slug,rating,alfredo,almond,american,appetizer,appetizers,apple,asiancuisine,asparagus,...,udonnoodles,vanilla,vanillaicecream,vegan,vegetables,vegetarian,warm,whippedcream,winter,yeastdough
0,falafel,0.800,0,0,0,1,0,0,0,0,...,0,0,0,1,0,1,0,0,0,0
1,spamburger,0.000,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,bacon-fried-rice,0.600,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
3,chicken-fingers,0.350,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,apple-crisp,0.850,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,bacon-mac-and-cheese,0.785,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
96,chicken-alfredo-lasagna,0.655,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
97,classic-beef-lasagna,0.685,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
98,vegetarian-mushroom-lasagna,0.455,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0


In [22]:
features['bias'] = 1
features

,recipe_slug,rating,alfredo,almond,american,appetizer,appetizers,apple,asiancuisine,asparagus,...,vanilla,vanillaicecream,vegan,vegetables,vegetarian,warm,whippedcream,winter,yeastdough,bias
0,falafel,0.800,0,0,0,1,0,0,0,0,...,0,0,1,0,1,0,0,0,0,1
1,spamburger,0.000,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2,bacon-fried-rice,0.600,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,1
3,chicken-fingers,0.350,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,apple-crisp,0.850,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,bacon-mac-and-cheese,0.785,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
96,chicken-alfredo-lasagna,0.655,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
97,classic-beef-lasagna,0.685,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
98,vegetarian-mushroom-lasagna,0.455,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,1


In [23]:
features.to_csv('features.tsv', sep='\t', index=False)

Submit "features.tsv" in Gradescope.

## Part 3: Linear Preference Model

Use your feature and rating files to build a ridge regression model with ridge regression's regularization parameter $\alpha$ set to 1.


Hint: If you are using scikit-learn modeling classes, you should use `fit_intercept=False` since that intercept value will be redundant with the bias coefficient.

Hint: The estimate component of the bounds should match the previous estimate, so you should be able to just focus on the variance component of the bounds now.

In [29]:
from sklearn.linear_model import Ridge

X = features.drop(columns=['recipe_slug', 'rating'])
y = ratings['rating']

model = Ridge(alpha=1, fit_intercept=False)
model.fit(X, y)

coef = model.coef_

coef_per_feature = pd.DataFrame(
    {'recipe_tag': X.columns, 'coefficient': coef}
)
coef_per_feature

,recipe_tag,coefficient
0,alfredo,0.010412
1,almond,0.042751
2,american,0.018762
3,appetizer,0.013526
4,appetizers,0.038811
...,...,...
292,warm,0.039899
293,whippedcream,-0.005673
294,winter,0.008793
295,yeastdough,-0.007925


Save the coefficients of this model in a file "model.tsv" with columns "recipe_tag" and "coefficient".
Do not add anything for the `intercept_` attribute of a scikit-learn model; this will be covered by the coefficient for the bias column added in part 2.

In [30]:
coef_per_feature.to_csv('model.tsv', sep='\t', index=False)

Submit "model.tsv" in Gradescope.

## Part 4: Recipe Estimates

Use the recipe model to estimate the score of every recipe.
Save these estimates to a file "estimates.tsv" with columns recipe_slug and score_estimate.

In [40]:
estimates = model.predict(X)

recipe_estimates = pd.DataFrame(
    {'recipe_slug': features['recipe_slug'], 'score_estimate': estimates}
)

recipe_estimates

,recipe_slug,score_estimate
0,falafel,0.744353
1,spamburger,0.043499
2,bacon-fried-rice,0.593638
3,chicken-fingers,0.380885
4,apple-crisp,0.855665
...,...,...
95,bacon-mac-and-cheese,0.757474
96,chicken-alfredo-lasagna,0.644588
97,classic-beef-lasagna,0.702823
98,vegetarian-mushroom-lasagna,0.559919


Submit "estimates.tsv" in Gradescope.

## Part 5: LinUCB Bounds

Calculate the upper bounds of LinUCB using data corresponding to trying every recipe once and receiving the rating in "ratings.tsv" as the reward.
Keep the ridge regression regularization parameter at 1, and set LinUCB's $\alpha$ parameter to 2.
Save these upper bounds to a file "bounds.tsv" with columns recipe_slug and score_bound.

In [ ]:
# YOUR CHANGES HERE

...

Submit "bounds.tsv" in Gradescope.

## Part 6: Make Online Recommendations

Implement LinUCB to make 100 recommendations starting with no data and using the same parameters as in part 5.
One recommendation should be made at a time and you can break ties arbitrarily.
After each recommendation, use the rating from part 1 as the reward to update the LinUCB data.
Record the recommendations made in a file "recommendations.tsv" with columns "recipe_slug", "score_bound", and "reward".
The rows in this file should be in the same order as the recommendations were made.

Hint: do not remove recipes after each recommendation.
Repeating recommendations is expected.

In [ ]:
# YOUR CHANGES HERE

...

Submit "recommendations.tsv" in Gradescope.

## Part 7: Acknowledgments

Make a file "acknowledgments.txt" documenting any outside sources or help on this project.
If you discussed this assignment with anyone, please acknowledge them here.
If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for.
If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy.
If no acknowledgements are appropriate, just write none in the file.


Submit "acknowledgments.txt" in Gradescope.

## Part 8: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.


Submit "project.ipynb" in Gradescope.